# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using the [Croissant schema](https://mlcommons.org/croissant/) and is published at the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)
# Access metadata as an object (not via dictionary subscripting)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

We will print the list of record sets, and for each, print the associated fields and columns along with their `@id` values.

In [ ]:
# List record sets with their @id
print('Available Record Sets (with their @id):')
for record_set in metadata.record_sets:
    print(f"- {record_set.id} (name: {record_set.name})")
    print("  Fields (@id, name, dataType):")
    for field in record_set.fields:
        # Print field id, name, and data type
        print(f"    - {field.id}: {field.name} ({field.data_type})")
    print("  Columns (@id, name):")
    if hasattr(record_set, 'columns') and record_set.columns is not None:
        for column in record_set.columns:
            print(f"    - {column.id}: {column.name}")
    print()

## 3. Data Extraction
We'll extract the data from each record set into a pandas DataFrame for further analysis. All record set and field references use their full `@id` values per Croissant convention.

We will display field IDs and sample data from one chosen record set.

In [ ]:
# Gather all record set @id strings
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set '{record_set_id}' with columns:")
    print(df.columns.tolist())

# For demonstration, pick the first available record set
if record_sets:
    main_record_set_id = record_sets[0]
    main_df = dataframes[main_record_set_id]
    print(f"\nSample data from {main_record_set_id}:")
    display(main_df.head())
else:
    print("No record sets found in dataset.")

## 4. Exploratory Data Analysis (EDA)
Let's perform common preprocessing steps such as filtering numeric fields, normalizing, handling missing data, or grouping by categorical fields. All field references use their `@id`.

In [ ]:
# We'll select the first numeric field available in the main record set
import numpy as np

if record_sets:
    rs_meta = None
    for r in metadata.record_sets:
        if r.id == main_record_set_id:
            rs_meta = r
            break

    # Identify the first numeric field (Float or Integer)
    numeric_field = None
    group_field = None
    for field in rs_meta.fields:
        if field.data_type is not None and field.data_type.lower() in ['float', 'integer', 'number']:
            numeric_field = field.id
            break
    
    # Choose a group_field as the first non-numeric
    for field in rs_meta.fields:
        if field.id != numeric_field and field.data_type not in ['Float', 'Integer', 'Number']:
            group_field = field.id
            break
    
    if numeric_field:
        col = numeric_field
        df = main_df
        # Convert column to numeric
        df[col] = pd.to_numeric(df[col], errors='coerce')
        threshold = df[col].mean()  # Use mean as a threshold
        filtered_df = df[df[col] > threshold]
        print(f"Filtered records where {col} > {threshold:.2f}:")
        print(filtered_df[[col]].head())
        # Normalize
        filtered_df[col + '_normalized'] = (filtered_df[col] - filtered_df[col].mean()) / (filtered_df[col].std() + 1e-7)
        print(f"\nNormalized sample of {col}:")
        print(filtered_df[[col, col + '_normalized']].head())

        # Group by group_field if present
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[col].mean().reset_index(name=f'{col}_mean')
            print(f"\nMean of {col} grouped by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA in main record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Let's visualize distributions or relationships between fields (histogram of a numeric field, or a bar plot by group).

We'll use matplotlib for basic plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Grouped bar plot if group_field available
    if group_field and group_field in main_df.columns:
        grouped_df = main_df.groupby(group_field)[numeric_field].mean().reset_index(name=f'{numeric_field}_mean')
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field, y=f'{numeric_field}_mean')
        plt.title(f"Mean of {numeric_field} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean of {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion

- We have successfully loaded the FAIR² Mental Health Survey dataset using the machine-actionable Croissant schema and the `mlcroissant` library.
- Using only `@id` identifiers for entities, we extracted and processed record set data, performed EDA and visualized main features.
- This dataset enables public health and mental health research for Kilifi County, Kenya, with explicit field and data type documentation for robust, reproducible AI/ML workflows.